# Practical Solutions 1–6

Run `!pip install optuna shap -q` and upload `loan_applications.csv` first. Each solution is self-contained.

## Solution 1 — Practical 1: Hierarchical clustering & DBSCAN on borrower behaviour

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)

feats = ["income_monthly", "bureau_score", "credit_utilization",
         "emi_to_income", "num_delinquencies_2y"]
sample = df[feats].dropna().sample(600, random_state=42)
X = StandardScaler().fit_transform(sample)      # distance-based -> scaling is mandatory

# ---------- 1. Dendrogram to choose the number of clusters
Z = linkage(X, method="ward")
plt.figure(figsize=(13, 4))
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=28)
plt.axhline(28, color="red", ls="--", lw=2)
plt.title("Ward dendrogram (last 30 merges) — the tall vertical gap tells you where to cut")
plt.ylabel("merge distance")
plt.tight_layout(); plt.show()

# ---------- 2. Agglomerative at a few K values
print("K   silhouette")
for k in [2, 3, 4, 5, 6]:
    lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
    print(f"{k}   {silhouette_score(X, lab):.3f}")

agg = AgglomerativeClustering(n_clusters=4, linkage="ward")
sample["hier_cluster"] = agg.fit_predict(X)

# ---------- 3. Choose eps for DBSCAN with a k-distance plot
nn = NearestNeighbors(n_neighbors=10).fit(X)
dist, _ = nn.kneighbors(X)
kdist = np.sort(dist[:, -1])
plt.figure(figsize=(9, 3.5))
plt.plot(kdist, color="#1F4E79", lw=2)
plt.axhline(1.2, color="red", ls="--")
plt.xlabel("points sorted by distance to 9th nearest neighbour")
plt.ylabel("9-NN distance")
plt.title("k-distance plot - set eps at the knee")
plt.tight_layout(); plt.show()

# ---------- 4. DBSCAN
db = DBSCAN(eps=1.2, min_samples=10).fit(X)
sample["dbscan_cluster"] = db.labels_
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise = int((db.labels_ == -1).sum())
print(f"\nDBSCAN found {n_clusters} cluster(s) and flagged {n_noise} points as noise "
      f"({n_noise/len(sample):.1%} of borrowers)")
print("Try other settings and watch what happens:")
for e in [0.9, 1.0, 1.1, 1.2, 1.3]:
    d2 = DBSCAN(eps=e, min_samples=10).fit(X)
    k2 = len(set(d2.labels_)) - (1 if -1 in d2.labels_ else 0)
    print(f"  eps={e}: {k2} cluster(s), {(d2.labels_ == -1).mean():.1%} noise")

# ---------- 5. Compare all three
km = KMeans(4, n_init=10, random_state=42).fit_predict(X)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lab, name in zip(axes, [km, sample["hier_cluster"], sample["dbscan_cluster"]],
                         ["K-Means (K=4)", "Agglomerative ward (K=4)", "DBSCAN"]):
    ax.scatter(sample["bureau_score"], sample["emi_to_income"], c=lab, cmap="Set1", s=18)
    ax.set_xlabel("bureau_score"); ax.set_ylabel("emi_to_income"); ax.set_title(name)
plt.tight_layout(); plt.show()

# ---------- 6. Who are the DBSCAN outliers? This is the business payoff.
outliers = sample[sample["dbscan_cluster"] == -1]
print("\n--- Profile: DBSCAN noise points vs everyone else ---")
print(pd.concat([outliers[feats].mean().rename("outliers"),
                 sample[sample["dbscan_cluster"] != -1][feats].mean().rename("clustered")],
                axis=1).round(2))

# ---------- 7. Default rate per cluster
sample["default"] = df.loc[sample.index, "default"]
print("\nDefault rate by hierarchical cluster:")
print(sample.groupby("hier_cluster")["default"].agg(["mean", "count"]).round(3))
print("\nDefault rate among DBSCAN outliers:",
      round(df.loc[outliers.index, "default"].mean(), 3))

# INTERPRETATION - read this carefully, it is the most useful result of the module
#
# 1. Silhouette peaks at K=2 (0.186) and every value is LOW. Low silhouette means
#    the borrowers do not form well-separated natural groups - they are one
#    continuous cloud. Hierarchical clustering still returned 4 clusters, because
#    you ASKED for 4. It will always give you what you asked for.
#
# 2. DBSCAN refuses to play along. At every sensible eps it reports ONE cluster
#    plus a set of outliers. That is not a failure - it is the correct answer, and
#    it is information K-Means structurally cannot give you: "there are no density
#    gaps in this population."
#
# 3. So the business value here is the NOISE label, not the clusters. Those ~12%
#    of borrowers sit in sparse regions of feature space: unusual combinations the
#    scorecard has thin evidence for. They have roughly double the delinquency
#    count of the main body. That is a manual-underwriting queue, not an
#    automated decision.
#
# 4. The hierarchical clusters ARE still useful for pricing: default rates run
#    from 5.8% (cluster 1) to 38.5% (cluster 3). Weak separation does not mean
#    zero value - it means do not oversell the segments as "customer types".

## Solution 2 — Practical 2: PCA on the credit-bureau block

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)

bureau = ["bureau_score", "credit_utilization", "num_credit_lines", "num_inquiries_6m",
          "num_delinquencies_2y", "oldest_account_months", "revolving_balance",
          "total_debt", "debt_to_income", "income_monthly", "emi", "emi_to_income",
          "loan_amount", "interest_rate"]

data = df[bureau + ["default"]].dropna()
X, y = data[bureau], data["default"]

# ---------- 1. Correlation: is there anything to compress?
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), cmap="RdBu_r", center=0, annot=False)
plt.title("If these columns were independent, PCA would have nothing to do")
plt.tight_layout(); plt.show()

# ---------- 2. Scale, then fit PCA
X_scaled = StandardScaler().fit_transform(X)
pca = PCA().fit(X_scaled)
cum = np.cumsum(pca.explained_variance_ratio_)

for i, (ev, cv) in enumerate(zip(pca.explained_variance_ratio_, cum), 1):
    print(f"PC{i:<2} variance {ev:.3f}   cumulative {cv:.3f}")

k90 = int(np.argmax(cum >= 0.90)) + 1
print(f"\n{k90} of {len(bureau)} components retain 90% of the variance.")

# ---------- 3. Scree + cumulative
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, len(bureau)+1), pca.explained_variance_ratio_, color="#1F4E79")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance explained")
axes[0].set_title("Scree plot")
axes[1].plot(range(1, len(bureau)+1), cum, "o-", color="#1F4E79")
axes[1].axhline(0.90, color="red", ls="--")
axes[1].set_xlabel("components kept"); axes[1].set_ylabel("cumulative variance")
axes[1].set_title("Cumulative variance")
plt.tight_layout(); plt.show()

# ---------- 4. What do the components MEAN? Read the loadings.
loadings = pd.DataFrame(pca.components_[:3].T, index=bureau, columns=["PC1","PC2","PC3"])
print("\nTop drivers of PC1:")
print(loadings["PC1"].abs().sort_values(ascending=False).head(5).round(3))
print("\nTop drivers of PC2:")
print(loadings["PC2"].abs().sort_values(ascending=False).head(5).round(3))

# ---------- 5. Visualise in 2D
Z = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(8, 5))
plt.scatter(Z[y==0,0], Z[y==0,1], s=6, alpha=.3, c="#2E7D57", label="repaid")
plt.scatter(Z[y==1,0], Z[y==1,1], s=6, alpha=.4, c="#C0392B", label="defaulted")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("14 columns -> 2. PCA never saw the labels.")
plt.tight_layout(); plt.show()

# ---------- 6. Does PCA help a downstream model? MEASURE IT, do not assume.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
plain = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("sc", StandardScaler()),
                  ("m", LogisticRegression(max_iter=2000))])
withpca = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler()),
                    ("pca", PCA(n_components=k90)),
                    ("m", LogisticRegression(max_iter=2000))])
for name, p in [("all 14 features", plain), (f"PCA to {k90} components", withpca)]:
    s = cross_val_score(p, Xtr, ytr, cv=5, scoring="roc_auc").mean()
    print(f"{name:<28} CV ROC-AUC {s:.4f}")

# INTERPRETATION
# PCA is compression, not magic. It usually costs a little accuracy because it
# discards low-variance directions that may still carry signal. You use it when
# you need speed, fewer columns, decorrelated inputs, or a 2-D picture - NOT as a
# reflex before every model. And note PC1 is unreadable to a credit officer:
# "component 1 was too high" is not a legally usable reason for a rejection.

## Solution 3 — Practical 3: Hyperparameter tuning three ways

In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.stats import loguniform, randint
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)
X = df.drop(columns=["application_id", "default"])
y = df["default"]
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]

pre = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                      ("s", StandardScaler())]), num),
    ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                      ("o", OneHotEncoder(handle_unknown="ignore"))]), cat)])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
cv = StratifiedKFold(3, shuffle=True, random_state=42)
pipe = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(random_state=42))])

# ---------- 0. Baseline: untouched defaults
base = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=-1).mean()
print(f"DEFAULTS            CV AUC {base:.4f}")

# ---------- 1. GridSearchCV — exhaustive
grid = {"model__learning_rate": [0.03, 0.1],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 60]}
t0 = time.time()
gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1).fit(Xtr, ytr)
print(f"GRID  ({len(gs.cv_results_['params'])} fits x3 folds, {time.time()-t0:.0f}s) "
      f"CV AUC {gs.best_score_:.4f}")
print("   ", gs.best_params_)

# ---------- 2. RandomizedSearchCV — same budget, continuous ranges
space = {"model__learning_rate": loguniform(0.005, 0.3),
         "model__max_leaf_nodes": randint(8, 64),
         "model__min_samples_leaf": randint(10, 150),
         "model__l2_regularization": loguniform(1e-3, 10),
         "model__max_iter": randint(100, 500)}
t0 = time.time()
rs = RandomizedSearchCV(pipe, space, n_iter=25, cv=cv, scoring="roc_auc",
                        random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"RANDOM (25 fits x3 folds, {time.time()-t0:.0f}s) CV AUC {rs.best_score_:.4f}")
print("   ", {k: (round(v, 4) if isinstance(v, float) else v)
              for k, v in rs.best_params_.items()})

# ---------- 3. Optuna — Bayesian
# In Colab run this first:  !pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 64),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 150),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
        max_iter=trial.suggest_int("max_iter", 100, 500),
        random_state=42))])
    return cross_val_score(p, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=-1).mean()

t0 = time.time()
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=25, show_progress_bar=False)
print(f"OPTUNA (25 trials, {time.time()-t0:.0f}s) CV AUC {study.best_value:.4f}")
print("   ", {k: (round(v, 4) if isinstance(v, float) else v)
              for k, v in study.best_params.items()})

# ---------- 4. Final honest score on the untouched test set
final = Pipeline([("pre", pre),
                  ("model", HistGradientBoostingClassifier(random_state=42,
                                                           **study.best_params))]).fit(Xtr, ytr)
print(f"\nTEST AUC of the Optuna model: "
      f"{roc_auc_score(yte, final.predict_proba(Xte)[:,1]):.4f}")

# ---------- 5. Optuna tells you which hyperparameters mattered
print("\nHyperparameter importance:")
for k, v in optuna.importance.get_param_importances(study).items():
    print(f"  {k:<22} {v:.3f}")

# INTERPRETATION
# Grid search burned its budget on 8 rigid combinations. Random and Optuna
# explored continuous ranges and found better models with the same number of fits.
# Note the CV score is ALWAYS a little optimistic relative to the test score -
# you picked the winner using those folds, so they are no longer neutral.

## Solution 4 — Practical 4: Pipelines, custom transformers, and leakage

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
import joblib

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)
X = df.drop(columns=["application_id", "default"])
y = df["default"]

# ---------- 1. A custom transformer the EASY way: FunctionTransformer
def add_ratios(d):
    d = d.copy()
    d["debt_service_ratio"] = d["emi"] / d["income_monthly"].clip(lower=1)
    d["balance_per_line"]   = d["revolving_balance"] / d["num_credit_lines"].clip(lower=1)
    d["credit_age_years"]   = d["oldest_account_months"] / 12
    return d

ratio_maker = FunctionTransformer(add_ratios)
print(add_ratios(X).filter(like="ratio").head(3))

# ---------- 2. A custom transformer the PROPER way: BaseEstimator + TransformerMixin
# Use this when the transformer must LEARN something in fit() - here, the training
# medians used to cap outliers. Learning it in fit() is what keeps it leak-free.
class OutlierCapper(BaseEstimator, TransformerMixin):
    """Cap numeric columns at the p1/p99 learned FROM THE TRAINING FOLD ONLY."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        Xn = pd.DataFrame(X).select_dtypes(include=np.number)
        self.columns_ = Xn.columns
        self.lo_ = Xn.quantile(self.lower)
        self.hi_ = Xn.quantile(self.upper)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.columns_:
            X[c] = X[c].clip(self.lo_[c], self.hi_[c])
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)

# ---------- 3. Assemble the full pipeline
X_fe = add_ratios(X)
num = X_fe.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X_fe.columns if c not in num]

pre = ColumnTransformer([
    ("num", Pipeline([("cap", OutlierCapper()),
                      ("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])

full = Pipeline([("ratios", ratio_maker),
                 ("pre", pre),
                 ("model", HistGradientBoostingClassifier(random_state=42))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
full.fit(Xtr, ytr)
print("\nPipeline steps:", [n for n, _ in full.steps])
print("Test AUC:", round(
    __import__("sklearn").metrics.roc_auc_score(yte, full.predict_proba(Xte)[:,1]), 4))

# ---------- 4. Tune the WHOLE pipeline, preprocessing included
grid = {"pre__num__imp__strategy": ["median", "mean"],
        "pre__num__cap__upper": [0.99, 0.995],
        "model__learning_rate": [0.05, 0.12]}
gs = GridSearchCV(full, grid, cv=StratifiedKFold(3, shuffle=True, random_state=42),
                  scoring="roc_auc", n_jobs=-1).fit(Xtr, ytr)
print("\nBest pipeline params:", gs.best_params_)
print("Best CV AUC:", round(gs.best_score_, 4))

# ---------- 5. The leakage demonstration: 200 rows, 2000 noise features, RANDOM labels
rng = np.random.default_rng(42)
Xn = rng.normal(size=(200, 2000))
yn = rng.integers(0, 2, 200)
cvk = StratifiedKFold(5, shuffle=True, random_state=42)

X_selected_first = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)   # PEEKED
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected_first,
                        yn, cv=cvk, scoring="roc_auc").mean()
honest = cross_val_score(Pipeline([("sel", SelectKBest(f_classif, k=20)),
                                   ("m", LogisticRegression(max_iter=1000))]),
                         Xn, yn, cv=cvk, scoring="roc_auc").mean()
print(f"\nThere is NO signal in this data. Truth = 0.500")
print(f"  selection before CV (leaky) : {leaky:.3f}   <- invented from nothing")
print(f"  selection inside pipeline   : {honest:.3f}")

# ---------- 6. Serialise the ENTIRE pipeline, never just the model
joblib.dump(gs.best_estimator_, "credit_pipeline.joblib")
reloaded = joblib.load("credit_pipeline.joblib")
raw_applicant = Xte.iloc[[0]]                 # raw, unprocessed, straight from the form
print("\nReloaded pipeline scores a RAW row:",
      round(reloaded.predict_proba(raw_applicant)[0, 1], 4))

# INTERPRETATION
# The reloaded object accepts a raw DataFrame. That is the entire point: the
# imputation values, the caps, the scaler statistics and the encoder categories
# all travelled with the model. Save only the estimator and you must reimplement
# every one of those steps in production - which is how training/serving skew
# is born.

## Solution 5 — Practical 5: Persistence, a scoring service, and monitoring

In [ ]:
import numpy as np
import pandas as pd
import joblib, json, hashlib, datetime
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)
X = df.drop(columns=["application_id", "default"]); y = df["default"]
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]
pre = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
    ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                      ("o", OneHotEncoder(handle_unknown="ignore"))]), cat)])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
pipe = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(
    learning_rate=0.0129, max_leaf_nodes=16, min_samples_leaf=97,
    l2_regularization=1.617, max_iter=426, random_state=42))]).fit(Xtr, ytr)
test_auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])

# ---------- 1. Save the pipeline
joblib.dump(pipe, "credit_model_v1.joblib")

# ---------- 2. Save a MODEL CARD next to it. The file alone is not enough.
model_card = {
    "model_name": "credit_default_risk",
    "version": "1.0.0",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "algorithm": "HistGradientBoostingClassifier inside a sklearn Pipeline",
    "training_rows": int(len(Xtr)),
    "features_expected": list(X.columns),
    "target": "default (1 = defaulted within the performance window)",
    "test_roc_auc": round(float(test_auc), 4),
    "default_rate_train": round(float(ytr.mean()), 4),
    "intended_use": "Rank applicants for manual underwriting priority.",
    "out_of_scope": "Not for automated rejection without human review.",
    "known_limitations": [
        "Trained on applicants who were APPROVED under the old policy - "
        "we never observe how rejected applicants would have behaved.",
        "Gender correlates with income in this population; monitor group metrics."],
}
with open("model_card_v1.json", "w") as f:
    json.dump(model_card, f, indent=2)
print(json.dumps(model_card, indent=2)[:600], "...")

# ---------- 3. What a prediction service looks like (this is the whole thing)
class CreditScoringService:
    """Everything a FastAPI endpoint would wrap. Load once, score many."""
    def __init__(self, model_path, card_path):
        self.pipeline = joblib.load(model_path)
        self.card = json.load(open(card_path))
        self.expected = self.card["features_expected"]

    def _validate(self, payload: dict):
        missing = [c for c in self.expected if c not in payload]
        if missing:
            raise ValueError(f"Missing required fields: {missing[:5]}")

    def predict(self, payload: dict) -> dict:
        self._validate(payload)
        row = pd.DataFrame([payload])[self.expected]
        p = float(self.pipeline.predict_proba(row)[0, 1])
        band = "HIGH" if p >= 0.50 else "MEDIUM" if p >= 0.20 else "LOW"
        return {"default_probability": round(p, 4),
                "risk_band": band,
                "model_version": self.card["version"],
                "scored_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}

service = CreditScoringService("credit_model_v1.joblib", "model_card_v1.json")
applicant = Xte.iloc[0].to_dict()
print("\nAPI response:", json.dumps(service.predict(applicant), indent=2))

# ---------- 4. Drift monitoring with PSI (Population Stability Index)
def psi(expected, actual, buckets=10):
    """PSI < 0.10 stable | 0.10-0.25 watch | > 0.25 investigate now."""
    expected, actual = pd.Series(expected).dropna(), pd.Series(actual).dropna()
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, buckets + 1)))
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual,   bins=edges)[0] / len(actual)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

# Simulate next quarter's applicants: the marketing team pushed into a lower-income segment
rng = np.random.default_rng(7)
next_q = Xte.copy()
next_q["income_monthly"] = next_q["income_monthly"] * rng.uniform(0.55, 0.85, len(next_q))

print("\nPSI report (train vs incoming traffic):")
for col in ["income_monthly", "bureau_score", "emi_to_income", "credit_utilization"]:
    v = psi(Xtr[col], next_q[col])
    flag = "INVESTIGATE" if v > 0.25 else "watch" if v > 0.10 else "stable"
    print(f"  {col:<22} PSI {v:6.3f}   {flag}")

old = pipe.predict_proba(Xte)[:, 1].mean()
new = pipe.predict_proba(next_q)[:, 1].mean()
print(f"\nMean predicted risk: was {old:.3f}, now {new:.3f} "
      f"({(new-old)/old:+.1%}) - the portfolio got riskier before any label arrived.")

# INTERPRETATION
# Notice the order in which you learn about trouble: the INPUT distribution shifts
# first (PSI), the SCORE distribution shifts next, and the actual default labels
# arrive 6-12 months later. If you only monitor AUC you are flying blind for most
# of a year. That lag is the single most important operational fact in credit ML.

## Solution 6 — Practical 6: Fairness audit and explainability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, roc_auc_score

df = pd.read_csv("loan_applications.csv").drop_duplicates().reset_index(drop=True)
X = df.drop(columns=["application_id", "default"]); y = df["default"]
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]
pre = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
    ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                      ("o", OneHotEncoder(handle_unknown="ignore"))]), cat)])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
pipe = Pipeline([("pre", pre), ("model", HistGradientBoostingClassifier(
    learning_rate=0.0129, max_leaf_nodes=16, min_samples_leaf=97,
    l2_regularization=1.617, max_iter=426, random_state=42))]).fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
print("Overall test AUC:", round(roc_auc_score(yte, proba), 4))

# ---------- 1. Group fairness metrics
def group_report(y_true, y_pred, groups):
    rows = []
    for g in sorted(pd.Series(groups).unique()):
        m = (groups == g)
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m], labels=[0,1]).ravel()
        rows.append({"group": g, "n": int(m.sum()),
                     "actual_default_rate": y_true[m].mean(),
                     "selection_rate": y_pred[m].mean(),
                     "TPR_recall": tp/(tp+fn) if (tp+fn) else np.nan,
                     "FPR": fp/(fp+tn) if (fp+tn) else np.nan,
                     "precision": tp/(tp+fp) if (tp+fp) else np.nan})
    return pd.DataFrame(rows).round(3)

for attr in ["gender", "region"]:
    rep = group_report(yte.values, pred, Xte[attr].values)
    print(f"\n--- Fairness report by {attr} ---")
    print(rep.to_string(index=False))
    di = rep["selection_rate"].min() / rep["selection_rate"].max()
    print(f"Disparate impact ratio (min/max selection rate): {di:.3f}",
          "  PASSES the 4/5 rule" if di >= 0.8 else "  FAILS the 4/5 rule")

# ---------- 2. Does removing the protected attribute fix it? (Spoiler: no)
X2 = X.drop(columns=["gender"])
X2tr, X2te = X2.loc[Xtr.index], X2.loc[Xte.index]
num2 = [c for c in num if c != "gender"]; cat2 = [c for c in cat if c != "gender"]
pre2 = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num2),
    ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                      ("o", OneHotEncoder(handle_unknown="ignore"))]), cat2)])
blind = Pipeline([("pre", pre2), ("model", HistGradientBoostingClassifier(
    learning_rate=0.0129, max_leaf_nodes=16, min_samples_leaf=97,
    l2_regularization=1.617, max_iter=426, random_state=42))]).fit(X2tr, ytr)
pred_blind = (blind.predict_proba(X2te)[:, 1] >= 0.5).astype(int)
rep_blind = group_report(yte.values, pred_blind, Xte["gender"].values)
print("\n--- 'Gender-blind' model, still scored BY gender ---")
print(rep_blind.to_string(index=False))
print("The disparity does not vanish: income and employment_type are proxies.")

# ---------- 3. Global explainability: permutation importance
perm = permutation_importance(pipe, Xte, yte, n_repeats=5, random_state=42,
                              scoring="roc_auc", n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features by permutation importance (drop in AUC when shuffled):")
print(imp.head(10).round(4))

plt.figure(figsize=(9, 5))
imp.head(12).sort_values().plot(kind="barh", color="#1F4E79")
plt.xlabel("mean drop in ROC-AUC when the column is shuffled")
plt.title("Global explainability - what the model relies on")
plt.tight_layout(); plt.show()

# ---------- 4. Local explainability with SHAP
# In Colab run this first:  !pip install shap -q
import shap
pre_fitted = pipe.named_steps["pre"]
model = pipe.named_steps["model"]
feat_names = [n.split("__", 1)[1] for n in pre_fitted.get_feature_names_out()]
Xte_t = pre_fitted.transform(Xte)

explainer = shap.TreeExplainer(model)
sv = np.array(explainer.shap_values(Xte_t[:300]))
if sv.ndim == 3:
    sv = sv[..., 1]

riskiest = int(np.argmax(proba[:300]))
contrib = pd.Series(sv[riskiest], index=feat_names).sort_values(key=abs, ascending=False)
print(f"\nApplicant scored {proba[riskiest]:.1%} - top reasons:")
print(contrib.head(6).round(3))

shap.summary_plot(sv, Xte_t[:300], feature_names=feat_names, show=False, max_display=12)
plt.title("SHAP summary - direction AND magnitude, per feature")
plt.tight_layout(); plt.show()

# ---------- 5. Turn SHAP into an adverse-action reason, which is the legal requirement
reasons = contrib[contrib > 0].head(4)
print("\nADVERSE ACTION NOTICE (auto-drafted):")
print("Your application was declined. The main factors were:")
for i, (f, v) in enumerate(reasons.items(), 1):
    print(f"  {i}. {f.replace('_', ' ')}")

# INTERPRETATION
# 1. Dropping the protected column does not make a model fair - it just makes the
#    bias harder to measure. Keep the attribute for AUDITING even if you exclude
#    it from training.
# 2. The base rates differ between groups in the DATA. A model can be perfectly
#    calibrated and still produce unequal outcomes, because it is reproducing a
#    disparity that already exists in the world. Choosing which fairness
#    definition to enforce is a policy decision, not a technical one.
# 3. SHAP gives per-applicant reasons, which is what adverse-action regulation
#    actually requires. Feature importance alone does not satisfy it.